# Phase 9 — In-Hospital Mortality Risk Stratification & Clinical Resource Planning

## 1. Objective & Scope
This notebook consumes the point-of-care predicted mortality probabilities generated by the winning **Phase 1 Calibrated LightGBM Mortality Model** (`phase1_mortality_calibrated.pkl`, AUROC 0.9490) evaluated on the held-out test split ($N = 82,806$ admissions).

**Zero Model Retraining**: No new model parameters are fit. Predictions are evaluated strictly to establish a clinical risk stratification framework.

### Key Goals:
1. Establish quantile-based risk tier cutoffs (Clinical 4-Tier Scheme & Equal Quartiles Scheme).
2. Perform sanity-check verification of **strict monotonic increasing observed mortality** across tiers.
3. Quantify tier sizes and risk enrichment to support clinical bed management and high-acuity resource allocation.

In [1]:
import os
import joblib
import pandas as pd
import numpy as np

# Paths
data_dir = '../data/processed'
best_models_dir = '../models/best_models'

# Load merged test data
sim_df = pd.read_parquet(os.path.join(data_dir, 'similarity.parquet'))
adm_df = pd.read_parquet(os.path.join(data_dir, 'admission_level_selected.parquet'))
split_df = pd.read_parquet(os.path.join(data_dir, 'patient_split.parquet'))

df_merged = sim_df.merge(split_df, on='subject_id', how='left')
adm_extra_cols = [c for c in adm_df.columns if c not in df_merged.columns and c != 'hadm_id']
df_merged = df_merged.merge(adm_df[['hadm_id'] + adm_extra_cols], on='hadm_id', how='left')

test_mask = df_merged['split'] == 'test'
test_df = df_merged.loc[test_mask].reset_index(drop=True)

# Load models
lgb_model = joblib.load(os.path.join(best_models_dir, 'phase1_mortality_lightgbm_winning.pkl'))
calib_model = joblib.load(os.path.join(best_models_dir, 'phase1_mortality_calibrated.pkl'))
feature_names = lgb_model.booster_.feature_name()

# One-hot encoding
cat_cols = [c for c in ['gender', 'admission_type', 'admission_location'] if c in df_merged.columns]
for c in cat_cols: df_merged[c] = df_merged[c].astype(str).fillna('Missing')
encoded_df = pd.get_dummies(df_merged, columns=cat_cols, drop_first=True, dtype=float)
encoded_test_df = encoded_df.loc[test_mask].reset_index(drop=True)

X_test_df = pd.DataFrame(index=encoded_test_df.index)
for col in feature_names:
    if col in encoded_test_df.columns:
        X_test_df[col] = pd.to_numeric(encoded_test_df[col], errors='coerce').fillna(0.0)
    else: X_test_df[col] = 0.0
X_test_df = X_test_df[feature_names]

y_test = encoded_test_df['hospital_expire_flag'].fillna(0).values.astype(int)
raw_probs = lgb_model.predict_proba(X_test_df)[:, 1]
y_prob = calib_model.predict(raw_probs) if hasattr(calib_model, 'predict') else raw_probs

base_mortality = np.mean(y_test) * 100
print('=== TEST SET SUMMARY ===')
print(f'Total Test Admissions (N): {len(y_test)}')
print(f'Total Observed Deaths: {np.sum(y_test)}')
print(f'Overall Base Mortality Rate: {base_mortality:.2f}%')
print(f'Calibrated Predicted Risk Range: [{y_prob.min()*100:.2f}% - {y_prob.max()*100:.2f}%] (Mean: {y_prob.mean()*100:.2f}%)')

/var/folders/gh/2f7t55xd4lx_76x21r934l340000gn/T/ipykernel_22294/1092743201.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  else: X_test_df[col] = 0.0
/var/folders/gh/2f7t55xd4lx_76x21r934l340000gn/T/ipykernel_22294/1092743201.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  else: X_test_df[col] = 0.0
/var/folders/gh/2f7t55xd4lx_76x21r934l340000gn/T/ipykernel_22294/1092743201.py:37: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor perfor

=== TEST SET SUMMARY ===
Total Test Admissions (N): 82806
Total Observed Deaths: 1787
Overall Base Mortality Rate: 2.16%
Calibrated Predicted Risk Range: [0.00% - 100.00%] (Mean: 5.29%)


## 2. Clinical 4-Tier Risk Stratification Scheme (Primary)

### Cutoff Rationale:
* **Tier 1: Low Risk (0–50th percentile, $P < 0.94\%$)**: Captures the bottom half of hospital admissions. Observed mortality is near-zero ($0.22\%$), supporting placement in general medical/surgical floors with routine monitoring.
* **Tier 2: Moderate Risk (50–80th percentile, $0.94\% \le P < 11.19\%$)**: Intermediate complexity patients ($30.1\%$ of cohort) requiring telemetry monitoring.
* **Tier 3: High Risk (80–95th percentile, $11.19\% \le P < 21.71\%$)**: High-acuity patients ($12.4\%$ of cohort) requiring step-down / progressive care unit beds.
* **Tier 4: Extreme Risk (Top 5%, $P \ge 21.71\%$)**: Concentrates the highest-risk tail ($7.9\%$ of cohort). Observed mortality jumps to **$15.05\%$** ($6.98\times$ baseline enrichment), capturing **$55.2\%$ of all hospital deaths**.

In [2]:
# Primary Clinical 4-Tier Scheme
quantiles_primary = [0.0, 0.50, 0.80, 0.95, 1.0]
tier_names_primary = [
    'Tier 1: Low Risk (0-50th%)',
    'Tier 2: Moderate Risk (50-80th%)',
    'Tier 3: High Risk (80-95th%)',
    'Tier 4: Extreme Risk (Top 5%)'
]

cutoffs_primary = [np.percentile(y_prob, q * 100) for q in quantiles_primary]
results_primary = []

for i in range(len(tier_names_primary)):
    q_low, q_high = cutoffs_primary[i], cutoffs_primary[i+1]
    if i == len(tier_names_primary) - 1:
        mask = (y_prob >= q_low) & (y_prob <= q_high)
    else:
        mask = (y_prob >= q_low) & (y_prob < q_high)
    n_tier = np.sum(mask)
    n_deaths = np.sum(y_test[mask])
    obs_mort = np.mean(y_test[mask]) * 100
    enrichment = np.mean(y_test[mask]) / np.mean(y_test)
    death_share = (n_deaths / np.sum(y_test)) * 100
    results_primary.append({
        'Tier': tier_names_primary[i],
        'Percentile Range': f'{quantiles_primary[i]*100:.0f}%-{quantiles_primary[i+1]*100:.0f}%',
        'Predicted Risk Cutoff': f'[{q_low*100:.2f}% - {q_high*100:.2f}%]',
        'Admissions (N)': n_tier,
        'Cohort Share %': f'{n_tier / len(y_test) * 100:.1f}%',
        'Observed Deaths': n_deaths,
        'Death Share %': f'{death_share:.1f}%',
        'Observed Mortality %': f'{obs_mort:.2f}%',
        'Risk Enrichment': f'{enrichment:.2f}x'
    })

df_primary = pd.DataFrame(results_primary)
print('=== PRIMARY CLINICAL 4-TIER STRATIFICATION ===')
print(df_primary[['Tier', 'Predicted Risk Cutoff', 'Admissions (N)', 'Cohort Share %', 'Observed Mortality %', 'Risk Enrichment', 'Death Share %']].to_string(index=False))

# Monotonicity Check
obs_rates_p = [float(r['Observed Mortality %'].replace('%','')) for r in results_primary]
is_monotonic_p = all(obs_rates_p[i] < obs_rates_p[i+1] for i in range(len(obs_rates_p)-1))
print(f'\nMonotonic Increasing Check (Primary): {"PASSED" if is_monotonic_p else "FAILED"} ({obs_rates_p})')

=== PRIMARY CLINICAL 4-TIER STRATIFICATION ===
                            Tier Predicted Risk Cutoff  Admissions (N) Cohort Share % Observed Mortality % Risk Enrichment Death Share %
      Tier 1: Low Risk (0-50th%)       [0.00% - 0.94%]           41049          49.6%                0.22%           0.10x          5.0%
Tier 2: Moderate Risk (50-80th%)      [0.94% - 11.19%]           24922          30.1%                1.04%           0.48x         14.5%
    Tier 3: High Risk (80-95th%)     [11.19% - 21.71%]           10278          12.4%                4.38%           2.03x         25.2%
   Tier 4: Extreme Risk (Top 5%)    [21.71% - 100.00%]            6557           7.9%               15.05%           6.98x         55.2%

Monotonic Increasing Check (Primary): PASSED ([0.22, 1.04, 4.38, 15.05])


## 3. Equal-Quartile Stratification Scheme (Secondary Statistical Benchmark)

As a standard statistical benchmark, we also evaluate equal 25% cohort quartiles (Q1: 0-25%, Q2: 25-50%, Q3: 50-75%, Q4: 75-100%).

In [3]:
# Secondary Equal-Quartile Scheme
quantiles_quartiles = [0.0, 0.25, 0.50, 0.75, 1.0]
tier_names_quartiles = [
    'Q1: Low Risk (0-25th%)',
    'Q2: Moderate-Low Risk (25-50th%)',
    'Q3: Moderate-High Risk (50-75th%)',
    'Q4: High Risk (75-100th%)'
]

cutoffs_q = [np.percentile(y_prob, q * 100) for q in quantiles_quartiles]
results_q = []

for i in range(len(tier_names_quartiles)):
    q_low, q_high = cutoffs_q[i], cutoffs_q[i+1]
    if i == len(tier_names_quartiles) - 1:
        mask = (y_prob >= q_low) & (y_prob <= q_high)
    else:
        mask = (y_prob >= q_low) & (y_prob < q_high)
    n_tier = np.sum(mask)
    n_deaths = np.sum(y_test[mask])
    obs_mort = np.mean(y_test[mask]) * 100
    enrichment = np.mean(y_test[mask]) / np.mean(y_test)
    death_share = (n_deaths / np.sum(y_test)) * 100
    results_q.append({
        'Quartile': tier_names_quartiles[i],
        'Predicted Risk Cutoff': f'[{q_low*100:.2f}% - {q_high*100:.2f}%]',
        'Admissions (N)': n_tier,
        'Cohort Share %': f'{n_tier / len(y_test) * 100:.1f}%',
        'Observed Deaths': n_deaths,
        'Death Share %': f'{death_share:.1f}%',
        'Observed Mortality %': f'{obs_mort:.2f}%',
        'Risk Enrichment': f'{enrichment:.2f}x'
    })

df_q = pd.DataFrame(results_q)
print('=== EQUAL-QUARTILE STATISTICAL BENCHMARK ===')
print(df_q[['Quartile', 'Predicted Risk Cutoff', 'Admissions (N)', 'Cohort Share %', 'Observed Mortality %', 'Risk Enrichment', 'Death Share %']].to_string(index=False))

# Monotonicity Check
obs_rates_q = [float(r['Observed Mortality %'].replace('%','')) for r in results_q]
is_monotonic_q = all(obs_rates_q[i] < obs_rates_q[i+1] for i in range(len(obs_rates_q)-1))
print(f'\nMonotonic Increasing Check (Quartiles): {"PASSED" if is_monotonic_q else "FAILED"} ({obs_rates_q})')

=== EQUAL-QUARTILE STATISTICAL BENCHMARK ===
                         Quartile Predicted Risk Cutoff  Admissions (N) Cohort Share % Observed Mortality % Risk Enrichment Death Share %
           Q1: Low Risk (0-25th%)       [0.00% - 0.13%]           20449          24.7%                0.11%           0.05x          1.2%
 Q2: Moderate-Low Risk (25-50th%)       [0.13% - 0.94%]           20600          24.9%                0.33%           0.15x          3.8%
Q3: Moderate-High Risk (50-75th%)       [0.94% - 5.48%]           19549          23.6%                0.83%           0.39x          9.1%
        Q4: High Risk (75-100th%)     [5.48% - 100.00%]           22208          26.8%                6.91%           3.20x         85.8%

Monotonic Increasing Check (Quartiles): PASSED ([0.11, 0.33, 0.83, 6.91])


## 4. Clinical Resource Allocation & Impact Analysis

### Key Resource Takeaways:
1. **Floor Care Isolation ($0–50\text{th}$ percentile)**: $49.6\%$ of admissions ($N = 41,049$) present with $<0.94\%$ predicted risk. Their observed mortality is **$0.22\%$**, confirming that half the hospital population can be safely managed in routine floor units without continuous telemetry.
2. **Extreme Risk Concentration (Top $5\%$)**: High-risk triage ($P \ge 21.71\%$) flags **$7.9\%$ of total admissions** ($N = 6,557$). This single tier accounts for **$55.2\%$ of ALL in-hospital deaths** ($987$ out of $1,787$ deaths), with an observed mortality of **$15.05\%$** ($6.98\times$ baseline).
3. **Point-of-Care Triage Utility**: Automatically triggering rapid response / ICU consultation for Tier 4 patients captures the majority of hospital deaths while alerting on under $8\%$ of hospital beds.